Lab-12 - Generative Model with Attention
Objective
Implement a text generation model using attention to improve contextual understanding.
Scenario
Build a chatbot that generates replies from user input. Use attention so the model focuses

In [1]:
!pip install torch -q

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

In [3]:
pairs = [
    ("hello", "hi"),
    ("how are you", "i am fine"),
    ("what is your name", "i am a chatbot"),
    ("where do you live", "i live in Noida"),
]

In [4]:
from collections import Counter

words = []
for p in pairs:
    words += p[0].split()
    words += p[1].split()

vocab = {w:i+1 for i, w in enumerate(set(words))}
vocab["<pad>"] = 0

inv_vocab = {i:w for w,i in vocab.items()}

In [6]:
from collections import Counter

words = []
for p in pairs:
    words += p[0].split()
    words += p[1].split()

vocab = {w:i+1 for i, w in enumerate(set(words))}
vocab["<pad>"] = 0

inv_vocab = {i:w for w,i in vocab.items()}

In [8]:
def encode(sentence):
    return [vocab[w] for w in sentence.split()]

X = [encode(p[0]) for p in pairs]
Y = [encode(p[1]) for p in pairs]

In [9]:
from torch.nn.utils.rnn import pad_sequence

X = pad_sequence([torch.tensor(x) for x in X], batch_first=True)
Y = pad_sequence([torch.tensor(y) for y in Y], batch_first=True)

In [10]:
class AttentionModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)

        self.attn = nn.Linear(hidden_size, hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        embed = self.embedding(x)
        lstm_out, _ = self.lstm(embed)

        # Attention weights
        attn_weights = torch.softmax(self.attn(lstm_out), dim=1)

        # Context vector
        context = torch.sum(attn_weights * lstm_out, dim=1)

        output = self.fc(context)
        return output

In [11]:
model = AttentionModel(len(vocab), 32, 64)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 200

for epoch in range(epochs):
    optimizer.zero_grad()

    output = model(X)

    loss = criterion(output, Y[:,0])  # simple target

    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 3.0631186962127686
Epoch 20, Loss: 0.0004948169807903469
Epoch 40, Loss: 0.00011128821643069386
Epoch 60, Loss: 7.142659887904301e-05
Epoch 80, Loss: 5.706543743144721e-05
Epoch 100, Loss: 4.803714909940027e-05
Epoch 120, Loss: 4.1362571209901944e-05
Epoch 140, Loss: 3.6207533412380144e-05
Epoch 160, Loss: 3.200593710062094e-05
Epoch 180, Loss: 2.8638653020607308e-05


In [12]:
def predict(sentence):
    model.eval()
    x = torch.tensor([encode(sentence)])
    output = model(x)
    pred = torch.argmax(output, dim=1).item()
    return inv_vocab.get(pred, "")

print(predict("how are you"))

i
